(course-enzyme-molecular-mechanics)=

# Path B - Module 42: Molecular Mechanics (Energies)

How do we know if your new disulfide bridge actually stabilizes the enzyme? Geometry and force-field energy provide complementary information; a single potential energy does not establish physical or thermodynamic stability. 

In this module, you will learn to use the molecular mechanics engine of MolSysMT to calculate the energy profile of your industrial biocatalyst.

Before constructing a mechanical model, inspect the [effective model recipe](../../user/tools/molecular_mechanics/get_model_recipe.md). Stored unsupported tuning requests now refuse with named-field diagnostics; they are not silently applied. The recipe distinguishes physical controls from execution settings and does not prepare chemistry or change coordinates.


In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load our engineered complex
petase = msm.convert('pdb:6EQE', to_form='molsysmt.MolSys', selection='molecule_type=="protein"')
msm.build.add_missing_hydrogens(petase, pH=8.0)

bhet = msm.convert('C1=CC(=CC=C1C(=O)OCCO)C(=O)OCCO', from_form='string:smiles', to_form='molsysmt.MolSys')
msm.build.add_missing_hydrogens(bhet)

molsys = msm.merge([petase, bhet])

## 1. Calculating Global Potential Energy
The function `get_potential_energy()` uses a force field engine (OpenMM by default) to compute the energy of the current coordinates.

In [ ]:
# Calculate the total energy of the complex
energy = msm.molecular_mechanics.get_potential_energy(molsys)

print(f"Total Potential Energy: {energy}")

## 2. Non-Bonded Interaction Energy
The standard non-bonded force contribution between disjoint enzyme and substrate atom sets describes this prepared configuration. It is not the energy released on binding, a binding free energy or a prediction of binding strength. Parameterization and the placement of independently built molecules must be established before interpreting this merged-system example.

You may pass lists of atom selections: rows follow `selection` and columns follow `selection_2`, with a singleton matrix axis for a flat input. Repeated partitions or shared members within an axis are retained; atoms shared between axes are rejected. Empty sets produce typed zero entries. Source coordinates and engine parameters remain unchanged. See {ref}`the non-bonded energy tutorial <Tutorial_Get_non_bonded_potential_energy>` for units, shapes and periodic/custom-model limits. This note does not qualify the complete preparation scenario.


In [ ]:
# Calculate the non-bonded potential energy between enzyme and substrate
interaction = msm.molecular_mechanics.get_non_bonded_potential_energy(molsys, 
                                                                   selection='molecule_type=="protein"', 
                                                                   selection_2='molecule_type=="small molecule"')

print(f"Enzyme-Substrate Interaction Energy: {interaction}")

## 3. Stress Mapping (Forces)
Where is the enzyme most "stressed"? Large forces indicate regions that want to move or change conformation.

In [ ]:
# Get the forces on every atom
forces = msm.molecular_mechanics.get_forces(molsys)

print(f"Max force acting on the system: {forces.max()}")

--- 

## 🏆 Path B Challenge: The Energy Auditor

1. Calculate the energy of the **Wild-type PETase** vs your **Mutated PETase**.
2. Did the potential energy decrease (become more negative)? 
3. Identify the **atom index** that has the absolute highest force acting on it. 
4. Use `msm.get_label()` to see which residue that atom belongs to.

You have moved from geometry to physics! In **Module 43**, we will use these energies to relax the system: **Energy Minimization**.